# Chapter 3 Advanced Lab: Build Your Own Quantum TCO Calculator

**Applied Quantum Computing: A Leader's Guide to the Next Computing Revolution**  
Dr. Ernesto Lee | [Back to Book](https://liquid-books.github.io/applied-quantum-computing)

---

## Objective
Build a reusable Quantum TCO Calculator your organization can actually adopt.

The calculator reads a provider pricing config (updatable in one place, annually), accepts workload parameters, and outputs cost-per-useful-result across three pricing archetypes with a full sensitivity analysis.

## Why This Matters
Quantum cloud pricing changes every quarter. A CFO who hard-codes vendor prices into a model will rebuild it from scratch every year. This calculator is designed so the **vendor layer lives in a single config file** — update the prices once, the entire model re-prices automatically.

## Deliverable
- Working calculator with sensitivity table
- Markdown README explaining inputs
- **400-word memo:** Which workload types most reward migrating to reserved capacity?

## Estimated Time
60–120 minutes

---

In [ ]:
# Run this first — installs required packages (~15 seconds in Colab)
!pip install -q pandas matplotlib numpy tabulate

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate

print('Libraries loaded.')

## Part 1: Provider Pricing Config

This is the ONE place you update prices. Change these annually when Appendix F is refreshed.  
The rest of the calculator reads from this dict — nothing else needs changing.

In [ ]:
# ============================================================
# PROVIDER PRICING CONFIG — UPDATE THIS ANNUALLY
# Source: Appendix F (Market Snapshot) of the textbook
# Last updated: 2025
# ============================================================

PRICING_CONFIG = {
    'pay_per_shot': {
        'name': 'Pay-Per-Shot (e.g., IBM Quantum free tier / AWS Braket)',
        'cost_per_shot': 0.00035,       # USD per shot
        'monthly_minimum': 0,            # USD/month
        'queue_overhead_hours': 2.0,     # avg hours waiting in queue
        'max_circuit_depth': 100,
        'best_for': 'Exploration, <10K shots/month'
    },
    'pay_per_minute': {
        'name': 'Pay-Per-Minute (e.g., AWS Braket IQM / Azure Quantum)',
        'cost_per_minute': 0.075,        # USD per QPU-minute
        'shots_per_minute': 1200,        # avg shots executed per QPU-minute
        'monthly_minimum': 0,
        'queue_overhead_hours': 1.0,
        'best_for': 'Moderate workloads, variable shot counts'
    },
    'reserved': {
        'name': 'Reserved Capacity (enterprise contract)',
        'monthly_cost': 8500,            # USD/month for dedicated access block
        'shots_per_month': 5_000_000,    # shots included in reservation
        'queue_overhead_hours': 0.1,     # near-instant priority access
        'best_for': 'High-volume, latency-sensitive production workloads'
    }
}

print('Pricing config loaded.')
for k, v in PRICING_CONFIG.items():
    print(f"  {v['name']}")

## Part 2: Workload Parameters

Define your workload here. These are the inputs your CFO or procurement team would provide.

In [ ]:
# ============================================================
# WORKLOAD PARAMETERS — EDIT THESE FOR YOUR USE CASE
# ============================================================

WORKLOAD = {
    'name': 'Portfolio Optimization (6-asset Markowitz)',
    'shots_per_run': 10_000,             # shots per quantum circuit execution
    'runs_per_month': 500,               # how many times you run this per month
    'circuit_depth': 45,                 # gate depth of your circuit
    'expected_error_rate': 0.02,         # 2% 2Q gate error rate (typical NISQ)
    'error_mitigation_overhead': 3.0,    # shots multiplier for error mitigation
    'pre_post_processing_cost_month': 200,  # USD/month classical compute around quantum
    'value_per_useful_result': 15.0,     # USD value of one successful quantum result
}

# Derived quantities
total_shots_month = WORKLOAD['shots_per_run'] * WORKLOAD['runs_per_month']
mitigated_shots   = total_shots_month * WORKLOAD['error_mitigation_overhead']
useful_results    = WORKLOAD['runs_per_month'] * (1 - WORKLOAD['expected_error_rate'])

print(f"Workload: {WORKLOAD['name']}")
print(f"  Total shots/month (raw):      {total_shots_month:>12,.0f}")
print(f"  Total shots/month (mitigated):{mitigated_shots:>12,.0f}")
print(f"  Useful results/month:         {useful_results:>12,.0f}")

## Part 3: Compute Cost Across Three Archetypes

In [ ]:
def compute_tco(workload, pricing_config):
    """Compute monthly TCO and cost-per-useful-result for each pricing archetype."""
    results = []
    mitigated_shots = (workload['shots_per_run'] * workload['runs_per_month'] *
                       workload['error_mitigation_overhead'])
    useful_results = workload['runs_per_month'] * (1 - workload['expected_error_rate'])
    
    # Pay-per-shot
    p = pricing_config['pay_per_shot']
    quantum_cost = mitigated_shots * p['cost_per_shot']
    total_cost   = quantum_cost + workload['pre_post_processing_cost_month']
    results.append({
        'Archetype': 'Pay-Per-Shot',
        'Quantum Cost/mo': quantum_cost,
        'Total TCO/mo': total_cost,
        'Cost/Useful Result': total_cost / useful_results if useful_results > 0 else 0,
        'Queue Wait (avg hrs)': p['queue_overhead_hours'],
        'Best For': p['best_for']
    })
    
    # Pay-per-minute
    p = pricing_config['pay_per_minute']
    minutes_needed = mitigated_shots / p['shots_per_minute']
    quantum_cost   = minutes_needed * p['cost_per_minute']
    total_cost     = quantum_cost + workload['pre_post_processing_cost_month']
    results.append({
        'Archetype': 'Pay-Per-Minute',
        'Quantum Cost/mo': quantum_cost,
        'Total TCO/mo': total_cost,
        'Cost/Useful Result': total_cost / useful_results if useful_results > 0 else 0,
        'Queue Wait (avg hrs)': p['queue_overhead_hours'],
        'Best For': p['best_for']
    })
    
    # Reserved
    p = pricing_config['reserved']
    quantum_cost = p['monthly_cost']
    total_cost   = quantum_cost + workload['pre_post_processing_cost_month']
    results.append({
        'Archetype': 'Reserved Capacity',
        'Quantum Cost/mo': quantum_cost,
        'Total TCO/mo': total_cost,
        'Cost/Useful Result': total_cost / useful_results if useful_results > 0 else 0,
        'Queue Wait (avg hrs)': p['queue_overhead_hours'],
        'Best For': p['best_for']
    })
    
    return pd.DataFrame(results)

tco_df = compute_tco(WORKLOAD, PRICING_CONFIG)

# Format for display
display_df = tco_df.copy()
for col in ['Quantum Cost/mo', 'Total TCO/mo', 'Cost/Useful Result']:
    display_df[col] = display_df[col].apply(lambda x: f'${x:,.2f}')

print(tabulate(display_df[['Archetype','Quantum Cost/mo','Total TCO/mo',
                             'Cost/Useful Result','Queue Wait (avg hrs)']],
               headers='keys', tablefmt='grid', showindex=False))

## Part 4: Sensitivity Analysis

How does cost change when key parameters shift? This is the table your CFO actually wants.

In [ ]:
def sensitivity_analysis(base_workload, pricing_config):
    """Run sensitivity scenarios across key parameters."""
    scenarios = [
        ('Base Case',                 {**base_workload}),
        ('Error rate drops 10x',      {**base_workload, 'expected_error_rate': base_workload['expected_error_rate'] / 10}),
        ('Shots halved',              {**base_workload, 'shots_per_run': base_workload['shots_per_run'] // 2}),
        ('Volume doubles',            {**base_workload, 'runs_per_month': base_workload['runs_per_month'] * 2}),
        ('Mitigation overhead halved',{**base_workload, 'error_mitigation_overhead': base_workload['error_mitigation_overhead'] / 2}),
    ]
    
    rows = []
    for scenario_name, workload in scenarios:
        df = compute_tco(workload, pricing_config)
        for _, row in df.iterrows():
            rows.append({
                'Scenario': scenario_name,
                'Archetype': row['Archetype'],
                'Total TCO/mo': row['Total TCO/mo'],
                'Cost/Useful Result': row['Cost/Useful Result'],
            })
    
    return pd.DataFrame(rows)

sensitivity_df = sensitivity_analysis(WORKLOAD, PRICING_CONFIG)

# Pivot for readability
pivot = sensitivity_df.pivot(index='Scenario', columns='Archetype', values='Total TCO/mo')
pivot_fmt = pivot.applymap(lambda x: f'${x:,.0f}')
print('\nSensitivity Table — Monthly TCO by Scenario and Archetype:')
print(tabulate(pivot_fmt, headers='keys', tablefmt='grid'))

In [ ]:
# Visualize sensitivity
fig, ax = plt.subplots(figsize=(12, 6))

scenarios = pivot.index.tolist()
x = np.arange(len(scenarios))
width = 0.25
colors = ['steelblue', 'darkorange', 'gray']

for i, archetype in enumerate(pivot.columns):
    ax.bar(x + i * width, pivot[archetype], width,
           label=archetype, color=colors[i], alpha=0.85)

ax.set_xlabel('Scenario', fontsize=11)
ax.set_ylabel('Monthly TCO (USD)', fontsize=11)
ax.set_title(f'Quantum TCO Sensitivity Analysis\nWorkload: {WORKLOAD["name"]}',
             fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(scenarios, rotation=15, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('quantum_tco_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as quantum_tco_sensitivity.png')

## Deliverable: Your 400-Word Memo

Answer in the cell below:
1. **Which pricing archetype wins for your workload?** At what volume does reserved capacity break even vs. pay-per-shot?
2. **Which sensitivity lever matters most?** What single parameter change most reduces cost-per-useful-result?
3. **Recommendation for your CFO:** Under what conditions should your organization commit to reserved capacity?

---
**Target:** ~400 words. Use numbers from your sensitivity table.

**YOUR MEMO HERE:**

TO: CFO  
FROM: [Your Name]  
RE: Quantum Cloud Pricing Recommendation  
DATE: [Today's Date]

*[Write your 400-word memo here...]*